In [1]:
import numpy as np

In [3]:
str(np.random.rand(10).astype(np.uint8).dtype)

'uint8'

In [ ]:
from concurrent.futures import Executor, Future
from multiprocessing import Processor
from multiprocessing.shared_memory import SharedMemory
from pathlib import Path

import numpy as np
from decord import VideoReader


class AsyncVideoReader:
    def __init__(self, path: str | Path):
        video_reader = VideoReader(str(path), **kwargs)

        try:
            frame0 = video_reader[10].asnumpy()
            video_reader.seek(0)
        except IndexError:
            frame0 = video_reader[0].asnumpy()
            video_reader.seek(0)

        self._shape = (self._video_reader._num_frame, *frame0.shape)

        self._dtype = str(frame0.dtype)

        # TODO: Create process that runs the `AsyncVideoReader._process` staticmethod

    @staticmethod
    def _process(path, queue, <shared_memory_args>):
        """runs in another process to get frames asychronously and place them in a shared buffer"""
        video_reader = VideoReader(str(path), **kwargs)
        # TODO: shared memory opening stuff here

        while True:
            request = request.get()
            if request is None:
                break

            # read frame
            frame = video_reader[request].asnumpy()
            # TODO: create a new shared memory if the resulting frame is of a different shape than the existing shared memory
            # TODO: write frame to shared memory and signal that the result is ready so the Future can be resolved, call its callbacks, etc.
            
    @property
    def shape(self) -> tuple[int, ...]
        return self._shape

    @property
    def dtype(self) -> str:
        return self._dtype
        
    @property
    def ndim(self) -> int:
        return len(self._shape)

    def __getitem__(self, indices) -> Future:
        # sends a request for a new frame, returns a Future immediately

In [1]:
import os

# decord uses up all the RAM otherwise
os.environ["DECORD_EOF_RETRY_MAX"] = "128"
#os.environ["LD_PRELOAD"] = "/usr/lib/x86_64-linux-gnu/libcuda.so:/usr/lib/x86_64-linux-gnu/libnvcuvid.so:$LD_PRELOAD"

from concurrent.futures import Future
import multiprocessing
from multiprocessing import Process, Queue
from multiprocessing.shared_memory import SharedMemory
from pathlib import Path
import threading

import numpy as np
from decord import VideoReader

mp_ctx = multiprocessing.get_context("spawn")


def _reader_process(path: Path, kwargs: dict, shm_name: str, frame_shape: tuple,
             dtype: str, request_queue: Queue, response_queue: Queue):
    import os        
    os.environ["DECORD_EOF_RETRY_MAX"] = "128"
    vr = VideoReader(str(path))
    vr[10].asnumpy()
    vr.seek(0)
    dtype = np.dtype(dtype)

    shm = SharedMemory(name=shm_name)
    buf = np.ndarray(frame_shape, dtype=dtype, buffer=shm.buf)

    while True:
        request = request_queue.get(block=True)
        if request is None:
            break

        rid, index = request
        frame = vr[index].asnumpy()
        # vr.seek(0)

        if frame.shape != buf.shape or frame.dtype != dtype:
            shm.close()
            shm = SharedMemory(create=True, size=frame.nbytes)
            buf = np.ndarray(frame.shape, dtype=frame.dtype, buffer=shm.buf)
            dtype = frame.dtype

        np.copyto(buf, frame)
        response_queue.put((rid, frame.shape, str(frame.dtype), shm.name))

    shm.close()


class AsyncVideoReader:
    def __init__(self, path: str | Path, **kwargs):
        self._path = Path(path)
        self._kwargs = kwargs

        vr = VideoReader(str(self._path))
        try:
            frame0 = vr[10].asnumpy()
            vr.seek(0)
        except IndexError:
            frame0 = vr[0].asnumpy()
            vr.seek(0)

        self._shape = (len(vr), *frame0.shape)
        self._dtype = np.dtype(frame0.dtype)
        del vr

        self._shm = SharedMemory(create=True, size=frame0.nbytes)

        self._request_queue: Queue = mp_ctx.Queue()
        self._response_queue: Queue = mp_ctx.Queue()

        self._pending_rid: int = 0
        self._pending_future: Future | None = None
        self._lock = threading.Lock()

        self._worker = mp_ctx.Process(
            target=_reader_process,
            kwargs=dict(
                path=self._path,
                kwargs=self._kwargs,
                shm_name=self._shm.name,
                frame_shape=frame0.shape,
                dtype=str(self._dtype),
                request_queue=self._request_queue,
                response_queue=self._response_queue,
            ),
            daemon=True,
        )
        self._worker.start()

        self._listener = threading.Thread(target=self._listen, daemon=True)
        self._listener.start()

    def _listen(self):
        while True:
            msg = self._response_queue.get()
            if msg is None:
                break

            rid, frame_shape, dtype, shm_name = msg

            with self._lock:
                if rid != self._pending_rid:
                    continue

                if shm_name != self._shm.name:
                    self._shm.unlink()
                    self._shm.close()
                    self._shm = SharedMemory(name=shm_name)

                result = np.ndarray(frame_shape, dtype=dtype, buffer=self._shm.buf).copy()
                future = self._pending_future

            future.set_result(result)

    def __getitem__(self, index: int) -> Future:
        with self._lock:
            if self._pending_future is not None and not self._pending_future.done():
                self._pending_future.cancel()

            self._pending_rid += 1
            future = Future()
            self._pending_future = future

        self._request_queue.put((self._pending_rid, index))
        return future

    def shutdown(self, wait: bool = True):
        self._request_queue.put(None)
        if wait:
            self._worker.join()
            self._response_queue.put(None)
            self._listener.join()
        self._shm.unlink()
        self._shm.close()

    @property
    def shape(self) -> tuple[int, ...]:
        return self._shape

    @property
    def dtype(self) -> np.dtype:
        return self._dtype

    @property
    def ndim(self) -> int:
        return len(self._shape)

In [2]:
import fastplotlib as fpl

Unable to find extension: VK_EXT_physical_device_drm


Image(value=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x01,\x00\x00\x007\x08\x06\x00\x00\x00\xb6\x1bw\x99\x…

Valid,Device,Type,Backend,Driver
✅ (default),AMD Radeon RX 570 Series (RADV POLARIS10),DiscreteGPU,Vulkan,Mesa 22.3.6
✅,NVIDIA GeForce RTX 3080,DiscreteGPU,Vulkan,590.44.01
❗ limited,"llvmpipe (LLVM 15.0.6, 256 bits)",CPU,Vulkan,Mesa 22.3.6 (LLVM 15.0.6)
❌,"AMD Radeon RX 570 Series (polaris10, LLVM 15.0.6, DRM 3.49, 6.1.0-41-amd64)",Unknown,OpenGL,4.6 (Core Profile) Mesa 22.3.6


pygfx version from git (0.9.0) and __version__ (0.16.0) don't match.
To silence this warning, use a fully namespaced name.


In [3]:
from ipywidgets import IntSlider, VBox
from functools import partial

In [4]:
paths = sorted(Path("/home/kushal/data/gerbils/").glob("*.mp4"))

In [5]:
vrs = list()
for p in paths:
    vrs.append(AsyncVideoReader(p))

Traceback (most recent call last):
  File "<string>", line 1, in <module>
    from multiprocessing.spawn import spawn_main; spawn_main(tracker_fd=84, pipe_handle=90)
                                                  ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/usr/local/lib/python3.13/multiprocessing/spawn.py", line 132, in _main
    self = reduction.pickle.load(from_parent)
AttributeError: Can't get attribute '_reader_process' on <module '__main__' (<class '_frozen_importlib.BuiltinImporter'>)>
Traceback (most recent call last):
  File "<string>", line 1, in <module>
    from multiprocessing.spawn import spawn_main; spawn_main(tracker_fd=84, pipe_handle=99)
                                                  ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd

In [6]:
fig = fpl.Figure(shape=(2, 2))

images = list()
for subplot, vr in zip(fig, vrs):
    f = vr[0]
    images.append(subplot.add_image(f.result(5)))

def update_index(change):
    global images
    
    i = change["new"]
    futures = list()
    for vr, g in zip(vrs, images):
        future = vr[i]
        future.add_done_callback(partial(_update_graphic, g))
        futures.append(future)
        
    while all([f.running() for f in futures]):
        pass

def _update_graphic(g, fut: Future):
    if fut.cancelled():
        return
    g.data = fut.result()

slider = IntSlider(value=0, min=0, max=vrs[0].shape[0])
slider.observe(update_index, "value")

VBox([fig.show(), slider])

RFBOutputContext()

In [3]:
vr[100]

<Future at 0x7f6cc4321f90 state=pending>

In [4]:
f = _

In [8]:
f.add_done_callback(

<Future at 0x7f6cc4321f90 state=finished returned ndarray>

In [1]:
import numpy as np

In [25]:
z = np.random.rand(1600, 1600, 3).astype(np.uint8).copy()
a = np.random.rand(1600, 1600, 3).astype(np.uint8).copy()

In [31]:
%%timeit -n 100
z[:] = a[:]

131 μs ± 7.96 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [1]:
import fastplotlib as fpl
import numpy as np

Unable to find extension: VK_EXT_physical_device_drm


Image(value=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x01,\x00\x00\x007\x08\x06\x00\x00\x00\xb6\x1bw\x99\x…

Valid,Device,Type,Backend,Driver
✅ (default),AMD Radeon RX 570 Series (RADV POLARIS10),DiscreteGPU,Vulkan,Mesa 22.3.6
✅,NVIDIA GeForce RTX 3080,DiscreteGPU,Vulkan,590.44.01
❗ limited,"llvmpipe (LLVM 15.0.6, 256 bits)",CPU,Vulkan,Mesa 22.3.6 (LLVM 15.0.6)
❌,"AMD Radeon RX 570 Series (polaris10, LLVM 15.0.6, DRM 3.49, 6.1.0-41-amd64)",Unknown,OpenGL,4.6 (Core Profile) Mesa 22.3.6


pygfx version from git (0.9.0) and __version__ (0.16.0) don't match.
To silence this warning, use a fully namespaced name.


In [2]:
fig = fpl.Figure()

img = fig[0, 0].add_image(np.random.rand(100, 100))

fig.show()

RFBOutputContext()

JupyterRenderCanvas(css_height='300.0px', css_width='500.0px')

In [4]:
import pygfx

In [63]:
img.world_object.children[0].geometry.grid = pygfx.Texture((np.random.rand(100, 100) * 10).astype("float64").copy(), dim=2)

ValueError: A dtype of float64 is not supported for buffers, use a 32-bit variant instead.

In [59]:
img.world_object.children[0].geometry.grid.data

array([[0, 5, 8, ..., 7, 1, 6],
       [3, 2, 1, ..., 7, 4, 7],
       [3, 8, 6, ..., 3, 8, 1],
       ...,
       [7, 0, 1, ..., 2, 3, 7],
       [0, 4, 4, ..., 3, 3, 3],
       [7, 9, 3, ..., 8, 9, 3]], shape=(100, 100), dtype=uint16)

In [60]:
img.vmin, img.vmax = 0, 10

In [56]:
img.reset_vmin_vmax()

In [45]:
img.world_object.children[0].geometry.grid.update_full()

In [114]:
def start_coroutine(func):
    def start(indices, block=True, timeout=1):
        if block:
            print("sync")
            cr = func(indices, block, timeout)
            fut = cr.send(None)
            try:
                cr.send(fut + 1)
            except StopIteration:
                pass
        else:
            print("async")
            cr = func(indices, block, timeout)
            yielded_val = cr.send(None)
            return cr, yielded_val
    return start

In [115]:
@coroutine
def async_func(indices, block=True, timeout=1):
    print("called")
    if block:
        result = future.result(timeout)
    else:
        result = yield future
        
    print("result", result)

In [116]:
@start_coroutine
def async_func(indices, block=True, timeout=1):
    print("called")
    result = yield 1    
    print("result", result)

In [117]:
async_func(dict(), block=True)

sync
called
result 2


In [86]:
gen, val = async_func(dict())

called
result 2


ValueError: not enough values to unpack (expected 2, got 0)

In [71]:
gen.send(val + 1)

after yield 2


StopIteration: 